# Evaluating Model Outputs

We can evaluate a model's confidence in its results by using perplexity. Perplexity is a measure of uncertainty (surprise) that can be calculated by exponentiating the negative of the average of the logprobs. 

+ Perplexity can be used to assess the result of an individual model run.
+ It can also be used to compare the relative confidence of results between model runs. 
+ Perplexity is NOT intuitive - a perplexity value of e.g. 2.03 means nothing in a vacuum. 
+ Perplexity does not detect errors 

Low perplexity or high confidence does not guarantee accuracy, but it can be a helpful signal when paired with other evaluation metrics. (High perplexity = low average log prop = greater unexpectedness) 

Of course for there to be uncertainty and confidence there must be some kind of probability distribution of results and their associated probabilities

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [2]:
from utils.clients import get_client
from IPython.display import display, Markdown
import numpy as np

client = get_client()

In [3]:
prompts = [
    # Low perplexity: Clear topic, common structure, highly predictable vocabulary
    "Explain how photosynthesis works in simple terms.",
    # Medium preplexity: Narrative freedom, but familiar theme and constraints.
    "Write a short story about a traveler who realizes the journey mattered more than the destination.",
    # High perplexity: Abstract concept, creative freedom, unpredictable vocabulary
    "Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather."
]

In [4]:
def get_completion(
    input: list[dict[str, str]],
    model: str = "gpt-4o-mini",
    max_tokens=500,
    temperature=0,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "input": input,
        "max_output_tokens": max_tokens,
        "temperature": temperature,
        "tools": tools,
        "include": ["message.output_text.logprobs"] if logprobs else [],
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.responses.create(**params)
    return completion

In [5]:
for k, prompt in enumerate(prompts):
    API_RESPONSE = get_completion(
        [{"role": "user", "content": prompt}],
        model="gpt-4o-mini",
        logprobs=True,
    )
    token_data = API_RESPONSE.output[0].content[0].logprobs
    response_text = API_RESPONSE.output[0].content[0].text
    lp_values = [t.logprob for t in token_data]
    perplexity_score = np.exp(-np.mean(lp_values))

    rows = [
        f"| `{t.token.replace('|', chr(9474)).replace(chr(10), '↵').replace(chr(96), chr(39))}` "
        f"| {t.logprob:.4f} | {np.exp(t.logprob)*100:.1f}% |"
        for t in token_data
    ]
    table = "\n".join([
        "| Token | Logprob | Linear Prob |",
        "|:------|--------:|------------:|",
    ] + rows)

    display(Markdown(f"### Prompt {k+1}\n_{prompt}_"))
    display(Markdown(f"**Response:**\n\n{response_text}"))
    display(Markdown(table))
    display(Markdown(f"**Perplexity:** `{perplexity_score:.2f}`\n\n---"))

### Prompt 1
_Explain how photosynthesis works in simple terms._

**Response:**

Photosynthesis is the process that plants, algae, and some bacteria use to make their own food. Here’s how it works in simple terms:

1. **Sunlight**: Plants take in sunlight using a green pigment called chlorophyll, which is found in their leaves.

2. **Water**: Plants absorb water from the soil through their roots.

3. **Carbon Dioxide**: Plants take in carbon dioxide from the air through tiny openings in their leaves called stomata.

4. **Making Food**: Using the energy from sunlight, plants combine water and carbon dioxide to create glucose (a type of sugar) and oxygen. The glucose is used as food for energy and growth.

5. **Oxygen Release**: The oxygen produced during this process is released back into the air, which is essential for us and other living beings to breathe.

In summary, photosynthesis is how plants turn sunlight, water, and carbon dioxide into food and oxygen!

| Token | Logprob | Linear Prob |
|:------|--------:|------------:|
| `Photos` | -0.0793 | 92.4% |
| `ynthesis` | 0.0000 | 100.0% |
| ` is` | 0.0000 | 100.0% |
| ` the` | -0.0436 | 95.7% |
| ` process` | -0.0041 | 99.6% |
| ` that` | -0.2282 | 79.6% |
| ` plants` | -0.0049 | 99.5% |
| `,` | -0.5768 | 56.2% |
| ` algae` | -0.0261 | 97.4% |
| `,` | 0.0000 | 100.0% |
| ` and` | 0.0000 | 100.0% |
| ` some` | -0.0000 | 100.0% |
| ` bacteria` | -0.0001 | 100.0% |
| ` use` | -0.0000 | 100.0% |
| ` to` | 0.0000 | 100.0% |
| ` make` | -0.1376 | 87.1% |
| ` their` | -0.0067 | 99.3% |
| ` own` | -0.0789 | 92.4% |
| ` food` | -0.0000 | 100.0% |
| `.` | -0.5787 | 56.1% |
| ` Here` | -0.2042 | 81.5% |
| `’s` | -0.0001 | 100.0% |
| ` how` | -0.1002 | 90.5% |
| ` it` | 0.0000 | 100.0% |
| ` works` | -0.0000 | 100.0% |
| ` in` | -0.0238 | 97.6% |
| ` simple` | -0.0001 | 100.0% |
| ` terms` | -0.0015 | 99.8% |
| `:↵↵` | -0.0000 | 100.0% |
| `1` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Sun` | -0.6001 | 54.9% |
| `light` | -0.0003 | 100.0% |
| `**` | -0.0996 | 90.5% |
| `:` | -0.0000 | 100.0% |
| ` Plants` | -0.0010 | 99.9% |
| ` take` | -0.8827 | 41.4% |
| ` in` | -0.0006 | 99.9% |
| ` sunlight` | -0.0190 | 98.1% |
| ` using` | -0.4961 | 60.9% |
| ` a` | -0.1481 | 86.2% |
| ` green` | -0.0354 | 96.5% |
| ` pigment` | -0.0022 | 99.8% |
| ` called` | -0.0486 | 95.3% |
| ` chlor` | -0.0000 | 100.0% |
| `ophyll` | 0.0000 | 100.0% |
| `,` | -0.5955 | 55.1% |
| ` which` | -0.7341 | 48.0% |
| ` is` | -0.0001 | 100.0% |
| ` found` | -0.4720 | 62.4% |
| ` in` | -0.0136 | 98.7% |
| ` their` | -0.0025 | 99.8% |
| ` leaves` | -0.0000 | 100.0% |
| `.↵↵` | -0.0028 | 99.7% |
| `2` | 0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Water` | -0.3352 | 71.5% |
| `**` | -0.3896 | 67.7% |
| `:` | 0.0000 | 100.0% |
| ` Plants` | -0.6621 | 51.6% |
| ` absorb` | -0.0070 | 99.3% |
| ` water` | -0.0000 | 100.0% |
| ` from` | -0.0405 | 96.0% |
| ` the` | -0.0001 | 100.0% |
| ` soil` | -0.0142 | 98.6% |
| ` through` | -0.0000 | 100.0% |
| ` their` | -0.0000 | 100.0% |
| ` roots` | -0.0000 | 100.0% |
| `.↵↵` | -0.0009 | 99.9% |
| `3` | 0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Carbon` | -0.0005 | 100.0% |
| ` D` | -0.0052 | 99.5% |
| `ioxide` | -0.0000 | 100.0% |
| `**` | -0.0002 | 100.0% |
| `:` | 0.0000 | 100.0% |
| ` Plants` | -0.0479 | 95.3% |
| ` take` | -0.0411 | 96.0% |
| ` in` | -0.0004 | 100.0% |
| ` carbon` | -0.0043 | 99.6% |
| ` dioxide` | 0.0000 | 100.0% |
| ` from` | -0.0513 | 95.0% |
| ` the` | -0.0000 | 100.0% |
| ` air` | -0.0000 | 100.0% |
| ` through` | -0.0002 | 100.0% |
| ` tiny` | -0.0790 | 92.4% |
| ` openings` | -0.0113 | 98.9% |
| ` in` | -0.0189 | 98.1% |
| ` their` | -0.0003 | 100.0% |
| ` leaves` | 0.0000 | 100.0% |
| ` called` | -0.0066 | 99.3% |
| ` stom` | -0.0001 | 100.0% |
| `ata` | -0.0000 | 100.0% |
| `.↵↵` | -0.0000 | 100.0% |
| `4` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Making` | -1.0721 | 34.2% |
| ` Food` | -0.0044 | 99.6% |
| `**` | -0.0001 | 100.0% |
| `:` | -0.0001 | 100.0% |
| ` Using` | -0.0633 | 93.9% |
| ` the` | -0.4750 | 62.2% |
| ` energy` | -0.2022 | 81.7% |
| ` from` | -0.0000 | 100.0% |
| ` sunlight` | -0.0032 | 99.7% |
| `,` | -0.0000 | 100.0% |
| ` plants` | -0.0086 | 99.1% |
| ` combine` | -0.1625 | 85.0% |
| ` water` | -0.1828 | 83.3% |
| ` and` | -0.0001 | 100.0% |
| ` carbon` | 0.0000 | 100.0% |
| ` dioxide` | 0.0000 | 100.0% |
| ` to` | -0.0035 | 99.6% |
| ` create` | -0.3005 | 74.0% |
| ` glucose` | -0.0264 | 97.4% |
| ` (` | -0.0382 | 96.3% |
| `a` | -0.0003 | 100.0% |
| ` type` | -0.0120 | 98.8% |
| ` of` | 0.0000 | 100.0% |
| ` sugar` | -0.0000 | 100.0% |
| `)` | -0.0670 | 93.5% |
| ` and` | -0.0014 | 99.9% |
| ` oxygen` | -0.0004 | 100.0% |
| `.` | -0.1272 | 88.1% |
| ` The` | -0.8216 | 44.0% |
| ` glucose` | -0.3444 | 70.9% |
| ` is` | -0.5645 | 56.9% |
| ` used` | -0.4254 | 65.4% |
| ` as` | -0.2694 | 76.4% |
| ` food` | -0.0298 | 97.1% |
| ` for` | -0.0561 | 94.5% |
| ` energy` | -0.6942 | 49.9% |
| ` and` | -0.0146 | 98.5% |
| ` growth` | -0.0000 | 100.0% |
| `.↵↵` | -0.2817 | 75.5% |
| `5` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `O` | -0.1553 | 85.6% |
| `xygen` | -0.0000 | 100.0% |
| ` Release` | -0.0254 | 97.5% |
| `**` | -0.0000 | 100.0% |
| `:` | 0.0000 | 100.0% |
| ` The` | -0.7220 | 48.6% |
| ` oxygen` | -0.1745 | 84.0% |
| ` produced` | -0.0232 | 97.7% |
| ` during` | -0.1218 | 88.5% |
| ` this` | -0.0322 | 96.8% |
| ` process` | -0.0000 | 100.0% |
| ` is` | -0.0001 | 100.0% |
| ` released` | -0.0021 | 99.8% |
| ` back` | -0.5759 | 56.2% |
| ` into` | 0.0000 | 100.0% |
| ` the` | 0.0000 | 100.0% |
| ` air` | -0.0142 | 98.6% |
| `,` | -0.0043 | 99.6% |
| ` which` | -0.0014 | 99.9% |
| ` is` | -0.0144 | 98.6% |
| ` essential` | -0.9060 | 40.4% |
| ` for` | -0.0000 | 100.0% |
| ` us` | -0.8152 | 44.3% |
| ` and` | -0.1636 | 84.9% |
| ` other` | -0.0632 | 93.9% |
| ` living` | -0.0813 | 92.2% |
| ` beings` | -0.5460 | 57.9% |
| ` to` | -0.0299 | 97.1% |
| ` breathe` | -0.0000 | 100.0% |
| `.↵↵` | -0.0010 | 99.9% |
| `In` | -0.4752 | 62.2% |
| ` summary` | -0.2072 | 81.3% |
| `,` | -0.0060 | 99.4% |
| ` photos` | -0.0859 | 91.8% |
| `ynthesis` | 0.0000 | 100.0% |
| ` is` | -1.0838 | 33.8% |
| ` how` | -0.0876 | 91.6% |
| ` plants` | -0.0001 | 100.0% |
| ` turn` | -0.4117 | 66.3% |
| ` sunlight` | -0.0033 | 99.7% |
| `,` | -0.0298 | 97.1% |
| ` water` | -0.0001 | 100.0% |
| `,` | -0.0000 | 100.0% |
| ` and` | 0.0000 | 100.0% |
| ` carbon` | -0.0142 | 98.6% |
| ` dioxide` | 0.0000 | 100.0% |
| ` into` | 0.0000 | 100.0% |
| ` food` | -0.0059 | 99.4% |
| ` and` | -0.1470 | 86.3% |
| ` oxygen` | -0.0010 | 99.9% |
| `!` | -0.2167 | 80.5% |

**Perplexity:** `1.12`

---

### Prompt 2
_Write a short story about a traveler who realizes the journey mattered more than the destination._

...

**Perplexity:** `1.52`

---

### Prompt 3
_Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather._

**Response:**

Imagine a color that tastes like the fleeting moment when twilight kisses the horizon—a blend of soft lavender and deep indigo, like the gentle curve of a parabolic arc just before it meets the ground. It’s the taste of a cool breeze, whispering secrets of impending rain, where each drop is a note in a symphony of flavors.

This color is a fleeting equation, a delicate balance of sweet and tart, reminiscent of the way a sudden gust of wind can shift the temperature by a degree, leaving you both exhilarated and contemplative. It’s the tang of a lightning strike, sharp and electric, yet softened by the warmth of the setting sun, like the last rays of light bending through a prism, scattering into a spectrum of possibilities.

In that one second, it’s as if time itself is a fractal, infinitely complex yet beautifully simple, where every taste is a point on a graph, each one leading to the next in a perfect, ephemeral sequence. It’s a moment suspended in the atmosphere, a fleeting taste of potential, as if the universe is reminding you that beauty is often found in the briefest of equations.

| Token | Logprob | Linear Prob |
|:------|--------:|------------:|
| `Imagine` | -0.3916 | 67.6% |
| ` a` | -0.5273 | 59.0% |
| ` color` | -0.5832 | 55.8% |
| ` that` | -0.0116 | 98.8% |
| ` tastes` | -1.2298 | 29.2% |
| ` like` | -0.0004 | 100.0% |
| ` the` | -0.3145 | 73.0% |
| ` fleeting` | -0.1691 | 84.4% |
| ` moment` | -0.3356 | 71.5% |
| ` when` | -0.2264 | 79.7% |
| ` twilight` | -0.8572 | 42.4% |
| ` kisses` | -1.7112 | 18.1% |
| ` the` | -0.0295 | 97.1% |
| ` horizon` | -0.0814 | 92.2% |
| `—a` | -0.7873 | 45.5% |
| ` blend` | -1.5389 | 21.5% |
| ` of` | -0.0250 | 97.5% |
| ` soft` | -1.7821 | 16.8% |
| ` lavender` | -0.5382 | 58.4% |
| ` and` | -0.0124 | 98.8% |
| ` deep` | -0.2897 | 74.8% |
| ` ind` | -0.1258 | 88.2% |
| `igo` | -0.0000 | 100.0% |
| `,` | -0.3654 | 69.4% |
| ` like` | -1.0402 | 35.3% |
| ` the` | -0.4401 | 64.4% |
| ` gentle` | -0.5880 | 55.5% |
| ` curve` | -0.3851 | 68.0% |
| ` of` | -0.0000 | 100.0% |
| ` a` | -0.0298 | 97.1% |
| ` par` | -0.4181 | 65.8% |
| `abolic` | -0.0013 | 99.9% |
| ` arc` | -0.0447 | 95.6% |
| ` just` | -1.2011 | 30.1% |
| ` before` | -0.0148 | 98.5% |
| ` it` | -0.0154 | 98.5% |
| ` meets` | -1.5560 | 21.1% |
| ` the` | -0.2315 | 79.3% |
| ` ground` | -0.4999 | 60.7% |
| `.` | -0.0384 | 96.2% |
| ` It` | -0.6988 | 49.7% |
| `’s` | -0.4482 | 63.9% |
| ` the` | -0.6696 | 51.2% |
| ` taste` | -1.4158 | 24.3% |
| ` of` | -0.0003 | 100.0% |
| ` a` | -0.2441 | 78.3% |
| ` cool` | -0.8110 | 44.4% |
| ` breeze` | -0.0458 | 95.5% |
| `,` | -1.4452 | 23.6% |
| ` whisper` | -1.3336 | 26.4% |
| `ing` | -0.0001 | 100.0% |
| ` secrets` | -0.5757 | 56.2% |
| ` of` | -0.1810 | 83.4% |
| ` impending` | -1.5788 | 20.6% |
| ` rain` | -0.5812 | 55.9% |
| `,` | -0.0548 | 94.7% |
| ` where` | -1.6733 | 18.8% |
| ` each` | -0.5710 | 56.5% |
| ` drop` | -0.4160 | 66.0% |
| ` is` | -0.5838 | 55.8% |
| ` a` | -0.0646 | 93.7% |
| ` note` | -1.8342 | 16.0% |
| ` in` | -0.0360 | 96.5% |
| ` a` | -0.1460 | 86.4% |
| ` sym` | -0.4394 | 64.4% |
| `phony` | -0.0699 | 93.2% |
| ` of` | -0.0683 | 93.4% |
| ` flavors` | -2.4126 | 9.0% |
| `.↵↵` | -0.8154 | 44.2% |
| `This` | -0.1351 | 87.4% |
| ` color` | -0.3345 | 71.6% |
| ` is` | -0.9852 | 37.3% |
| ` a` | -0.8460 | 42.9% |
| ` fleeting` | -0.6499 | 52.2% |
| ` equation` | -0.8765 | 41.6% |
| `,` | -0.0862 | 91.7% |
| ` a` | -0.9314 | 39.4% |
| ` delicate` | -1.7362 | 17.6% |
| ` balance` | -0.0168 | 98.3% |
| ` of` | -0.8638 | 42.2% |
| ` sweet` | -0.3082 | 73.5% |
| ` and` | -0.0100 | 99.0% |
| ` tart` | -1.0646 | 34.5% |
| `,` | -0.0875 | 91.6% |
| ` reminiscent` | -1.0042 | 36.6% |
| ` of` | 0.0000 | 100.0% |
| ` the` | -0.7372 | 47.8% |
| ` way` | -1.5061 | 22.2% |
| ` a` | -0.4356 | 64.7% |
| ` sudden` | -0.7270 | 48.3% |
| ` gust` | -0.3481 | 70.6% |
| ` of` | -0.9272 | 39.6% |
| ` wind` | -0.0007 | 99.9% |
| ` can` | -0.8675 | 42.0% |
| ` shift` | -0.2599 | 77.1% |
| ` the` | -0.1414 | 86.8% |
| ` temperature` | -0.1968 | 82.1% |
| ` by` | -0.6225 | 53.7% |
| ` a` | -0.3627 | 69.6% |
| ` degree` | -0.5204 | 59.4% |
| `,` | -0.5989 | 54.9% |
| ` leaving` | -0.9261 | 39.6% |
| ` you` | -0.8423 | 43.1% |
| ` both` | -1.2183 | 29.6% |
| ` exhilar` | -1.2727 | 28.0% |
| `ated` | -0.0000 | 100.0% |
| ` and` | -0.0000 | 100.0% |
| ` contempl` | -0.3912 | 67.6% |
| `ative` | -0.0001 | 100.0% |
| `.` | -0.0014 | 99.9% |
| ` It` | -0.1483 | 86.2% |
| `’s` | -0.3709 | 69.0% |
| ` the` | -0.8678 | 42.0% |
| ` tang` | -1.2432 | 28.8% |
| ` of` | -0.0154 | 98.5% |
| ` a` | -1.7971 | 16.6% |
| ` lightning` | -1.7491 | 17.4% |
| ` strike` | -0.7094 | 49.2% |
| `,` | -0.4416 | 64.3% |
| ` sharp` | -0.2384 | 78.8% |
| ` and` | -0.3537 | 70.2% |
| ` electric` | -0.1860 | 83.0% |
| `,` | -0.0074 | 99.3% |
| ` yet` | -1.0596 | 34.7% |
| ` softened` | -0.7636 | 46.6% |
| ` by` | -0.0100 | 99.0% |
| ` the` | -0.0378 | 96.3% |
| ` warmth` | -1.1404 | 32.0% |
| ` of` | -0.0009 | 99.9% |
| ` the` | -0.6537 | 52.0% |
| ` setting` | -1.2001 | 30.1% |
| ` sun` | -0.0000 | 100.0% |
| `,` | -0.5865 | 55.6% |
| ` like` | -0.7674 | 46.4% |
| ` the` | -0.4786 | 62.0% |
| ` last` | -1.5559 | 21.1% |
| ` rays` | -1.6229 | 19.7% |
| ` of` | -0.5931 | 55.3% |
| ` light` | -0.0962 | 90.8% |
| ` bending` | -1.9310 | 14.5% |
| ` through` | -0.5576 | 57.3% |
| ` a` | -0.3468 | 70.7% |
| ` prism` | -0.0047 | 99.5% |
| `,` | -0.8311 | 43.6% |
| ` scattering` | -0.8585 | 42.4% |
| ` into` | -0.9396 | 39.1% |
| ` a` | -0.6847 | 50.4% |
| ` spectrum` | -1.6858 | 18.5% |
| ` of` | -0.7238 | 48.5% |
| ` possibilities` | -0.9606 | 38.3% |
| `.↵↵` | -0.1700 | 84.4% |
| `In` | -0.6785 | 50.7% |
| ` that` | -0.1660 | 84.7% |
| ` one` | -1.0746 | 34.1% |
| ` second` | -0.1704 | 84.3% |
| `,` | -0.0446 | 95.6% |
| ` it` | -0.3319 | 71.8% |
| `’s` | -0.3943 | 67.4% |
| ` as` | -0.9400 | 39.1% |
| ` if` | -0.0373 | 96.3% |
| ` time` | -0.8217 | 44.0% |
| ` itself` | -0.8614 | 42.3% |
| ` is` | -1.1598 | 31.4% |
| ` a` | -0.2148 | 80.7% |
| ` fract` | -0.4262 | 65.3% |
| `al` | 0.0000 | 100.0% |
| `,` | -0.4042 | 66.8% |
| ` infinitely` | -0.3985 | 67.1% |
| ` complex` | -0.5708 | 56.5% |
| ` yet` | -0.1363 | 87.3% |
| ` beautifully` | -0.6398 | 52.7% |
| ` simple` | -0.0118 | 98.8% |
| `,` | -0.3178 | 72.8% |
| ` where` | -1.7608 | 17.2% |
| ` every` | -0.7924 | 45.3% |
| ` taste` | -0.3128 | 73.1% |
| ` is` | -0.8162 | 44.2% |
| ` a` | -0.1522 | 85.9% |
| ` point` | -1.5941 | 20.3% |
| ` on` | -0.2542 | 77.6% |
| ` a` | -0.0660 | 93.6% |
| ` graph` | -0.1712 | 84.3% |
| `,` | -0.7605 | 46.7% |
| ` each` | -2.2027 | 11.1% |
| ` one` | -0.8682 | 42.0% |
| ` leading` | -1.3799 | 25.2% |
| ` to` | -0.3905 | 67.7% |
| ` the` | -0.3266 | 72.1% |
| ` next` | -0.0265 | 97.4% |
| ` in` | -0.6274 | 53.4% |
| ` a` | -0.4248 | 65.4% |
| ` perfect` | -1.8225 | 16.2% |
| `,` | -0.6345 | 53.0% |
| ` ephemeral` | -0.9478 | 38.8% |
| ` sequence` | -1.3171 | 26.8% |
| `.` | -0.1138 | 89.2% |
| ` It` | -0.9277 | 39.5% |
| `’s` | -0.3924 | 67.5% |
| ` a` | -0.6330 | 53.1% |
| ` moment` | -0.7063 | 49.3% |
| ` suspended` | -1.0643 | 34.5% |
| ` in` | -0.3557 | 70.1% |
| ` the` | -0.5762 | 56.2% |
| ` atmosphere` | -0.5603 | 57.1% |
| `,` | -0.1757 | 83.9% |
| ` a` | -0.8286 | 43.7% |
| ` fleeting` | -1.5235 | 21.8% |
| ` taste` | -1.1232 | 32.5% |
| ` of` | -0.4609 | 63.1% |
| ` potential` | -1.9079 | 14.8% |
| `,` | -0.4508 | 63.7% |
| ` as` | -1.2589 | 28.4% |
| ` if` | -1.4842 | 22.7% |
| ` the` | -0.1213 | 88.6% |
| ` universe` | -0.5723 | 56.4% |
| ` is` | -1.4218 | 24.1% |
| ` reminding` | -1.1014 | 33.2% |
| ` you` | -0.0621 | 94.0% |
| ` that` | -0.2734 | 76.1% |
| ` beauty` | -0.3439 | 70.9% |
| ` is` | -1.3651 | 25.5% |
| ` often` | -0.1711 | 84.3% |
| ` found` | -0.2284 | 79.6% |
| ` in` | -0.0165 | 98.4% |
| ` the` | -0.1350 | 87.4% |
| ` brief` | -0.9222 | 39.8% |
| `est` | -0.0192 | 98.1% |
| ` of` | -0.0335 | 96.7% |
| ` equations` | -1.2468 | 28.7% |
| `.` | -0.1789 | 83.6% |

**Perplexity:** `1.83`

---